# plate-redactor — Phase 3: evaluation

Recall-focused evaluation of the Phase-2 checkpoint against a dedicated
**hard-case** test set (tilt / dirt / occlusion / shadow / tiny / mixed, plus a
portrait–landscape orientation check). The headline gate is **overall recall
≥ 0.90** — a missed plate is a data leak; an over-redacted region is harmless.

Runs on Kaggle or Colab (a CPU box is fine — 600 small images infer quickly).
Needs the trained `models/best.pt` from Phase 2 (release asset / Kaggle output /
Drive — **never committed**).

## 1. Install dependencies

The `[eval]` extra pulls `ultralytics` (model) + `matplotlib` (sweep plot).

In [ ]:
%pip install -q ultralytics matplotlib

import ultralytics, torch
ultralytics.checks()
print('CUDA available:', torch.cuda.is_available())

## 2. Get the code + detect platform

In [ ]:
import os, sys, subprocess
from pathlib import Path

ON_KAGGLE = Path('/kaggle').exists()
ON_COLAB = (not ON_KAGGLE) and ('google.colab' in sys.modules or Path('/content').exists())
print('Kaggle:', ON_KAGGLE, '| Colab:', ON_COLAB)

REPO_URL = 'https://github.com/Andre-Ehret/plate-redactor.git'
BRANCH = 'main'
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('/content')
REPO = WORK / 'plate-redactor'

if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, str(REPO)], check=True)
else:
    # Force to latest main so code fixes land on a plain re-run.
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', f'origin/{BRANCH}'], check=True)

%pip install -q -e {str(REPO)}
os.chdir(REPO)
head = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
print('cwd:', os.getcwd(), '| HEAD:', head)

## 3. Get the checkpoint (`models/best.pt`)

Point `MODEL` at the Phase-2 checkpoint. Pick the option that matches where you
stored it:
- **Kaggle Dataset** — attach it, then set the input path below.
- **GitHub release asset** — download it.
- **Google Drive** — mount and copy.

It is copied to `models/best.pt` (the script default).

In [ ]:
import shutil
MODEL = REPO / 'models' / 'best.pt'
MODEL.parent.mkdir(parents=True, exist_ok=True)

# --- Option A: attached Kaggle Dataset (edit the path) -------------------------
# src = Path('/kaggle/input/plate-detector-best/best.pt')
# shutil.copy2(src, MODEL)

# --- Option B: download a GitHub release asset (edit the URL) -------------------
# ASSET_URL = 'https://github.com/Andre-Ehret/plate-redactor/releases/download/v0.1/best.pt'
# subprocess.run(['curl', '-L', '-o', str(MODEL), ASSET_URL], check=True)

# --- Option C: Google Drive (Colab) --------------------------------------------
# from google.colab import drive; drive.mount('/content/drive')
# shutil.copy2('/content/drive/MyDrive/plate-redactor/best.pt', MODEL)

assert MODEL.exists(), f'{MODEL} missing — fill in one of the options above.'
print('checkpoint:', MODEL, f'({MODEL.stat().st_size/1e6:.1f} MB)')

## 4. Generate the hard-case test set

Seed 999 (not used in training) and the hardest augmentation settings. Attach
real backgrounds and add `--backgrounds /kaggle/input/<bg-dataset>` for a more
realistic test — omit for solid fallbacks (matches the released checkpoint).

In [ ]:
subprocess.run([
    sys.executable, 'src/eval/generate_test_set.py',
    '--out', 'data/test_hard', '--seed', '999',
    '--per-subset', '100', '--per-orientation', '50',
], check=True)

## 5. Evaluate (recall gate)

Operating conf 0.25 (low — recall bias). Writes the results table, the per-subset
+ overall metrics to `models/eval_results.json`, annotated false-negative images
to `models/eval_failures/`, and a draft `models/eval_report.md`.

In [ ]:
subprocess.run([
    sys.executable, 'src/eval/evaluate.py',
    '--model', str(MODEL),
    '--data', 'data/test_hard',
    '--conf', '0.25', '--imgsz', '320',
], check=True)

## 6. Confidence-threshold sweep

Sweeps conf 0.05–0.50, plots recall / precision / F1, marks the recommended
operating point (highest recall with precision ≥ 0.60) and records it back into
`models/eval_results.json`.

In [ ]:
subprocess.run([
    sys.executable, 'src/eval/threshold_sweep.py',
    '--model', str(MODEL), '--data', 'data/test_hard',
], check=True)

## 7. Inspect results

The sweep plot, the auto-drafted report, and a handful of annotated
false-negative images (GT red, predictions green) for visual failure analysis.

In [ ]:
import json
from IPython.display import Image as IPyImage, Markdown, display

res = json.loads((REPO / 'models' / 'eval_results.json').read_text())
print('overall:', res['overall'])
print('gates:  ', res['gates']['results'], '| passed:', res['gates']['passed'])
print('recommended conf:', res.get('threshold_sweep', {}).get('recommended_conf'))

sweep = REPO / 'models' / 'threshold_sweep.png'
if sweep.exists():
    display(IPyImage(filename=str(sweep)))

display(Markdown((REPO / 'models' / 'eval_report.md').read_text()))

fails = sorted((REPO / 'models' / 'eval_failures').rglob('*.jpg'))[:6]
print(f'\nShowing {len(fails)} of the false-negative cases:')
for f in fails:
    print(f.parent.name, '/', f.name)
    display(IPyImage(filename=str(f)))